# Demo 5 — The coach, up close

The coach answers questions about this course from **the pages in your clone**.
There is no model and no key. It is a search, and you can see every step.

In ten minutes:

1. ask a question, and open the page the answer came from
2. see why that page won
3. watch it say *"not in these pages"* — and watch it fail to say it
4. find a question it gets wrong, and work out why
5. measure it
6. turn a wrong answer into your first pull request

Runs offline.

In [1]:
# Setup. Works from anywhere inside the course checkout.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").exists():
    raise SystemExit(f"No course found above {Path.cwd()}. Open this inside your checkout.")
sys.path.insert(0, str(REPO_ROOT / "src"))
print("ready")

ready


## 1. Ask

`coach()` prints the passages that answer a question. Each one starts with the
page title and, in brackets, the **page id**.

In [2]:
from bootcamp_agent.coach import ask, coach

coach("where do I submit my work", top_k=1)

--- Handing work in  [unit0/how-to-submit]
# Handing work in

Every unit with a notebook can be handed in. Week 0 is a record of the work and
is never marked. **Every session carries marks**, scored from what its checks
print. Two sessions are assistant-driven, so nothing can re-run them and the
output you saved is the evidence the marks rest on — save the notebook before
you submit.

## What each command does

Nothing here is magic, and none of it touches the course repository.


The page id is a path. `unit0/how-to-submit` is the file
`units/en/unit0/how-to-submit.mdx`, and on the course site it is the same path.
**Always open the page.** The passage is a piece of it, and the rest of the page
is usually what you need next.

In [3]:
answer = ask("where do I submit my work", top_k=1)
page_id = answer.passages[0].chunk.doc_id

print("file:", f"units/en/{page_id}.mdx")
print("site:", f"https://gecko-academy.github.io/dev3pack-cohort-2026-09/{page_id}")

file: units/en/unit0/how-to-submit.mdx
site: https://gecko-academy.github.io/dev3pack-cohort-2026-09/unit0/how-to-submit


## 2. Why that page won

`ask()` returns the passages with a score. The score is not magic. It counts the
question's words in each passage, with three adjustments:

| Adjustment | What it does |
|---|---|
| **BM25** | a rare word counts more than a common one, and a long passage does not win only because it is long |
| **title weight** | a word that is also in the page title counts extra |
| **one per page** | the three answers come from three different pages |

In [4]:
def show(question, **switches):
    for passage in ask(question, **switches).passages:
        print(f"  {passage.score:6.2f}  {passage.chunk.doc_id}")

show("where do I submit my work")

    9.49  unit0/how-to-submit
    6.02  unit0/ask-your-assistant
    5.55  unit1/session-02-model-adapter/introduction


Now switch all three adjustments off. This is the plain word count you build in
session 6.

In [5]:
show("where do I submit my work", bm25=False, title_weight=0, one_per_page=False)

    5.26  unit0/ask-your-assistant
    5.26  unit0/how-to-submit
    5.26  unit1/session-02-model-adapter/introduction


With the adjustments off, the right page is **tied** with two others and loses
its first place. Each switch is one argument. Try them one at a time and see
what each one is worth.

## 3. "Not in these pages"

When no page shares a word with the question, the coach refuses. **That is a
correct answer.** A coach that always answers something is a coach you cannot
trust.

In [6]:
coach("xylophone quarterly dividend")

NOT IN THESE PAGES.
Nothing in your clone shares a word with that question. Either the
course does not cover it, or its week has not been published yet —
`git pull` on a Monday is what brings the next one.


Now a question the course does not cover, but that shares a word with it:

In [7]:
show("how do I deploy a Kubernetes ingress controller")

    4.77  bonus/b05-deploy-evaluate-teardown/introduction
    4.39  unit0/week3


It answers, because the pages it found say *deploy*. This is the dangerous case: an
answer that **looks** right. The coach has no idea that Kubernetes is not in the
course. It only knows that one word matched.

Nothing stops it today. A minimum score would, and that is an open
[good first issue](https://github.com/Gecko-Academy/gecko-ai-coach/issues?q=is%3Aissue+is%3Aopen+label%3A%22good+first+issue%22).

## 4. A question it gets wrong

Ask it the way a learner would:

In [8]:
show("how do I hand in a session")

    6.26  unit0/week1
    6.26  unit0/week2
    6.26  unit0/week3


Three week pages, all with the same score, and no `unit0/how-to-submit`.
Why? Look at the words the coach actually searched for:

In [9]:
from bootcamp_agent.coach import course_documents
from bootcamp_agent.retrieval import _tokens

question_words = _tokens("how do I hand in a session")
print("searched for:", question_words)

pages = {doc.doc_id: _tokens(doc.text) for doc in course_documents()}
for page_id in ["unit0/how-to-submit", "unit0/week1"]:
    counts = {word: pages[page_id].count(word) for word in question_words}
    print(f"  {page_id:22} {counts}")

searched for: ['hand', 'session']
  unit0/how-to-submit    {'hand': 3, 'session': 1}
  unit0/week1            {'hand': 1, 'session': 8}


The small words (`how`, `do`, `I`, `in`, `a`) are thrown away. Two words are left,
and **both are on both pages**. Read the counts:

- `hand` points to the submit page: 3 against 1
- `session` points to the week page: 8 against 1, because a week is a list of sessions

The coach scores passage by passage, and `session` wins. The three week pages tie
because they are built the same way.

The coach did exactly what it was built to do. What it cannot know is that *hand
in a session* means *submit*, and `submit` is the word the answer is about.

That is the most common reason any search misses: **the question and the page
use different words for the same thing.** There are fixes, and each one has a
cost: synonyms, word stems, embeddings.

## 5. Measure it

Every change to the coach is judged by one number: on 25 labelled questions, how
often is the right page in the top three?

In [10]:
from bootcamp_agent.coach_eval import measure

report = measure()

hit rate 21/25 = 84% (top-k)

missed:
  how much does a key cost
      got: unit1/session-05-deterministic-mini-agent/concepts-2, tracks/ship-it/introduction, tracks/depth/introduction
  can I run a model locally for free
      got: unit0/runtime-lanes, unit0/get-settled, unit1/session-05-deterministic-mini-agent/introduction
  what is the fake provider
      got: unit1/session-02-model-adapter/concepts-1, unit1/session-02-model-adapter/introduction, unit1/session-02-model-adapter/concepts-3
  what are structured outputs
      got: unit1/session-02-model-adapter/introduction, unit0/week1, unit1/session-04-bounded-tools/introduction


Your number depends on **which weeks are in your clone**: a page you do not have
yet cannot be found. That is why the same command can print a different number
on Monday.

The `missed:` list is the to-do list. Every line is a question a learner would
ask and the coach gets wrong.

## 6. Two retrievers, the same pages

The coach in this notebook and [@gecko_coach_bot](https://t.me/gecko_coach_bot)
on Telegram read **the same course pages**. The bot got two fixes this week.
Ask this notebook two questions:

In [ ]:
show("can I run a model locally for free")
show("how do I deploy a kubernetes ingress")

Now send the same two questions to the bot, from your phone.

| Question | This notebook | @gecko_coach_bot |
|---|---|---|
| `can I run a model locally for free` | three pages, and `unit0/local-model` is not one of them | `unit0/local-model` |
| `how do I deploy a kubernetes ingress` | three pages, each with a score. None of them is about Kubernetes. | a refusal |

**`locally` is not `local`.** To a word counter they are two different words, so
the page about running a model on your own machine never matched. A
contributor, Erol Tasci, fixed it: strip `-ly` from long words.
[He measured it before and after](https://github.com/Gecko-Academy/gecko-ai-coach/pull/21):
**76% → 80%** on 25 course questions, and nothing got worse.

**A score is not an answer.** This notebook always returns its top three, even
when the best page shares one common word with your question. The bot refuses
when its best page covers too few of your words.

Same pages, same question, different rules, different answers. **The number
above a passage counts shared words. It never says whether the passage answers
you.**

**Why these questions live here, and not on a course page.** Put them on a page
and that page contains every word of both questions. It would rank first for
both, and the bot would stop refusing. A page that quotes a question wins,
whether or not it answers it.

## Your turn

Nothing here is marked.

1. **Ask three questions you really had this week.** For each one, open the page.
   Did it answer you?
2. **Break the ranking on purpose.** Run `show()` on one of your questions with
   `bm25=False`, then with `title_weight=0`, then with `one_per_page=False`.
   Which switch mattered?
3. **Found a wrong answer? Hand it in — it takes ten minutes.** The coach lives in
   its own repository, [gecko-ai-coach](https://github.com/Gecko-Academy/gecko-ai-coach),
   and one command checks your question and saves it as a test:

   ```bash
   uv run ai-coach propose "how do I hand in a session" --page unit0/how-to-submit \
     --pages ../dev3pack-cohort-2026-09/units/en --write
   ```

   [CONTRIBUTING.md](https://github.com/Gecko-Academy/gecko-ai-coach/blob/main/CONTRIBUTING.md)
   walks through it step by step: fork, run the command, open the pull request.
4. **Want more?** Pick a
   [good first issue](https://github.com/Gecko-Academy/gecko-ai-coach/issues?q=is%3Aissue+is%3Aopen+label%3A%22good+first+issue%22),
   or read [bonus b06](../units/en/bonus/b06-improve-the-coach/introduction.mdx)
   to see why `measure(top_k=5)` scores higher and is **not** an improvement.